In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import sys, subprocess
root = Path('/content/drive/MyDrive/CameraTrap_Phase2')
run = root/'phase2_results/final_sprint_v1'
src = run/'code'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(src/'requirements.txt')],check=True)
sys.path.insert(0,str(src))
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader
from common import context, metrics, temperature, read, save, sha
from runtime import Classifier
from train import infer, Crops, samples
_,_,rows,classes = context(root)
policy = read(run/'policy.json')
entry = policy['species_classifier']
assert sha(entry['path']) == entry['sha256']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = Classifier(entry['path'],device).net.eval()
head = net.classifier[2]
net.classifier[2] = torch.nn.Identity()
out = run.parent/'species_head_trials'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out.mkdir(parents=True)
def extract(split):
    print('Extracting', split, flush=True)
    loader=DataLoader(Crops(samples(rows,classes,split),root),batch_size=24,num_workers=2)
    return infer(net,loader,device)
x,y=extract('train')
v,vy=extract('val_select')
layer=head[-1]
original_w=layer.weight.detach().cpu().numpy()[:14]
original_b=layer.bias.detach().cpu().numpy()[:14]
def score(w,b):
    return metrics(v@w.T+b,vy)
baseline=score(original_w,original_b)
best=baseline
chosen='original'
weights,bias=original_w,original_b
trials=[]
for balance in (None,'balanced'):
    fit=LogisticRegression(C=.1,class_weight=balance,max_iter=500).fit(x,y)
    assert np.array_equal(fit.classes_,np.arange(14))
    for mix in (.25,.5,1.):
        w=(1-mix)*original_w+mix*fit.coef_
        b=(1-mix)*original_b+mix*fit.intercept_
        m=score(w,b)
        trials.append(dict(balance=balance,mix=mix,accuracy=m['accuracy'],macro_f1=m['macro_f1']))
        if m['accuracy'] >= baseline['accuracy'] and m['macro_f1'] > best['macro_f1']:
            best,chosen,weights,bias=m,str((balance,mix)),w,b
cal,cy=extract('val_calibrate')
temp=temperature(cal@weights.T+bias,cy)
head_path=out/'species_head.npz'
np.savez(head_path,weight=weights.astype('float32'),bias=bias.astype('float32'))
save(out/'selection.json',dict(source_checkpoint=entry['sha256'],head_sha256=sha(head_path),classes=classes,
    temperature=temp,selected=chosen,baseline_validation=baseline,selected_validation=metrics(v@weights.T+bias,vy,temp),
    trials=trials,scope='Known-class crops, selection on val_select, temperature on val_calibrate'))
print('Selected:',chosen,'validation macro F1:',baseline['macro_f1'],'->',best['macro_f1'],'saved:',out)

import urllib.request, tempfile, shutil
revision='ced137fd3c6f06126b7fa4af213fbbb6d6d19fda'
local=out/'runtime'
local.mkdir()
for path in ('fast_candidate/candidate.py','fast_candidate/crop_sam.py','mask_refinement/combined.py','mask_refinement/bf16_encoder.py'):
    url=f'https://raw.githubusercontent.com/salehdev3/Capstone_5/{revision}/experiments/{path}'
    with urllib.request.urlopen(url,timeout=60) as stream:
        (local/Path(path).name).write_bytes(stream.read())
sys.path.insert(0,str(local))
from combined import CombinedPipeline
from evaluate import benchmark, mask_evaluate, reference, retained, frame_score
from legacy_evaluation import event_scores
from common import PLAN, aggregate, csv_save
from runtime import DEMO_IDS
import hashlib, tempfile, shutil
from PIL import Image
by_id = {r['id']:r for r in rows}
cache = Path(tempfile.mkdtemp(prefix='joint_cpu_'))
(cache/'images').mkdir()
def stage(ids):
    for ident in sorted(set(ids)):
        target = cache/'images'/f'{ident}.jpg'
        if not target.exists():
            shutil.copyfile(root/'images'/target.name,target)
            with Image.open(target) as im:
                assert hashlib.sha256(im.convert('RGB').tobytes()).hexdigest()==by_id[ident]['pixel_sha256']
policy.update(mask_mode='crop',encoder_precision='bf16_encoder',species_logit_offset=[0.]*14,
              species_temperature=temp,species_head=dict(path=str(head_path),sha256=sha(head_path)),
              runtime_revision=revision)
policy['mask']['sam']['size']=256
val_labels=reference(root,rows,'val_select')
stage(DEMO_IDS+[l.stem for l in val_labels])
profiles=[('float',224)]
flags=Path('/proc/cpuinfo').read_text()
if 'avx512_bf16' in flags or 'amx_bf16' in flags:
    profiles.insert(0,('bf16_encoder',256))
for precision,size in profiles:
    policy['encoder_precision']=precision
    policy['mask']['sam']['size']=size
    profile=out/f'{precision}_{size}'
    save(profile/'policy.json',policy)
    pipe=CombinedPipeline(root,cache,profile/'policy.json')
    quality=mask_evaluate(pipe,val_labels,by_id,cache,profile/'validation_masks')
    speed=benchmark(pipe,rows,cache,profile/'validation_cpu')
    print(precision,size,'Validation IoU:',quality['mean_iou'],'CPU p95:',speed['p95'],flush=True)
    if quality['mean_iou'] >= .70 and speed['p95'] <= 2:
        save(out/'policy.json',policy)
        break
if quality['mean_iou'] >= .70 and speed['p95'] <= 2:
    save(out/'freeze.json',dict(policy_sha256=sha(out/'policy.json'),independent_test=False))
    final_speed=benchmark(pipe,rows,cache,out/'final_cpu')
    selected_rows=retained(rows,'test')
    test_labels=reference(root,rows,'test')
    stage([r['id'] for r in selected_rows]+[l.stem for l in test_labels])
    frames={}
    counts=[]
    for i,r in enumerate(selected_rows,1):
        f=pipe.process_frame(cache/'images'/f"{r['id']}.jpg",r,with_masks=False)
        frames[r['id']]=f
        counts.append(frame_score(r,[o['box'] for o in f['objects']]))
        if i%50==0: print('Scored',i,'frames',flush=True)
    events,_=event_scores(rows,frames)
    csv_save(out/'event_counts.csv',events)
    d=aggregate(counts)
    q=mask_evaluate(pipe,test_labels,by_id,cache,out/'historical_masks')
    tx,ty=extract('test')
    species=metrics(tx@weights.astype('float32').T+bias.astype('float32'),ty,temp)
    save(out/'species_metrics.json',species)
    values=dict(animal_recall_min=d['animal_recall'],empty_removal_min=d['empty_removal'],
                species_macro_f1_min=species['macro_f1'],species_ece_max=species['ece'],
                mask_iou_min=q['mean_iou'],event_count_mae_max=float(np.mean([e['absolute_error'] for e in events])),
                p95_cpu_seconds_max=final_speed['p95'])
    gates=[dict(metric=k,value=values[k],target=t,met=bool(values[k]>=t if k.endswith('_min') else values[k]<=t))
           for k,t in PLAN['original_targets'].items()]
    csv_save(out/'acceptance.csv',gates)
    decision=dict(decision='GO' if all(g['met'] for g in gates) else 'NO-GO',acceptance=gates,
                  independent_test=False,policy_sha256=sha(out/'policy.json'))
    save(out/'decision.json',decision)
    print(decision)
else:
    print('Stopped at validation. Results saved in',out)
